In [1]:
from google.cloud import bigquery
from google.oauth2 import service_account
import pandas as pd
import pandas_gbq

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 10)

In [2]:
gcp_service_acc_file_aluno = "../secrets/serviceaccount.json"
gcp_id_do_projeto = "meu-projeto-eng-dados-512812"
my_gcp_cred = service_account.Credentials.from_service_account_file(gcp_service_acc_file_aluno)
my_gcp_client = bigquery.Client(credentials=my_gcp_cred)

In [3]:
df_silver = my_gcp_client.query(
    f"SELECT * FROM silver.complete_dc_marvel",
).to_dataframe(create_bqstorage_client=False)

In [ ]:
dim_film = (
    df_silver[["FILM", "GENRE", "RATING_MPA", "LANGUAGE", "BREAK_EVEN", "MCU", "MALE_FEMALE_LED", "PHASE"]]
    .drop_duplicates()
    .sort_values(by="FILM")#ORDENACAO POR FILME A-Z
    .reset_index(drop=True)#RESETA O INDEX ORIGINAL DO DF PARA 0,1,2,3,4,5,6,7,8,9
)

dim_film["ID_FILM"] = dim_film.index + 1

dim_film.head(5)

,FILM,GENRE,RATING_MPA,LANGUAGE,BREAK_EVEN,MCU,MALE_FEMALE_LED,PHASE,ID_FILM
0,Ant-Man,Superhero,PG-13,English,Success,True,Male,2.0,1
1,Ant-Man and the Wasp,Superhero,PG-13,English,Success,True,Co-starring,3.0,2
2,Ant-Man and the Wasp: Quantumania,Superhero,PG-13,English,Flop,True,Co-starring,5.0,3
3,Aquaman,Globetrotting Adventure,PG-13,English,Success,False,Male,NaN,4
4,Aquaman and the Lost Kingdom,Action Epic,PG-13,English,Flop,False,Male,NaN,5


In [11]:
dim_staff = (
    df_silver[["DIRECTOR", "WRITER", "STAR"]]
    .drop_duplicates()
    .sort_values(by=["DIRECTOR", "WRITER"])
    .reset_index(drop=True)
)

dim_staff["ID_STAFF"] = dim_staff.index + 1

dim_staff.head(3)

,DIRECTOR,WRITER,STAR,ID_STAFF
0,Alan Taylor,Christopher L Yost,"Chris Hemsworth, Natalie Portman, Tom Hiddleston",1
1,Andy Muschietti,Christina Hodson,"Ezra Miller, Michael Keaton, Sasha Calle",2
2,Andy Serkis,Kelly Marcel,"Tom Hardy, Woody Harrelson, Michelle Williams",3


In [15]:
dim_franchise = (
    df_silver[["FRANCHISE" , "CHARACTER_FAMILY"]]
    .drop_duplicates() 
    .sort_values(by=["FRANCHISE" , "CHARACTER_FAMILY"])
    .reset_index(drop=True)
)

dim_franchise["ID_FRANC"] = dim_franchise.index + 1 

dim_franchise.head(4)

,FRANCHISE,CHARACTER_FAMILY,ID_FRANC
0,DC,Aquaman,1
1,DC,Batman,2
2,DC,Birds of Prey,3
3,DC,Blue Beetle,4


In [17]:
dim_production = (
    df_silver[["PRODUCTION_COMPANY", "DISTRIBUTOR", "COUNTRY_ORIGIN", "FILMING_LOCATION"]]
    .drop_duplicates()
    .sort_values(by=["PRODUCTION_COMPANY", "DISTRIBUTOR", "COUNTRY_ORIGIN", "FILMING_LOCATION"])
    .reset_index(drop=True)
)

dim_production["ID_PRODUCTION"] = dim_production.index + 1

dim_production.head(3)

,PRODUCTION_COMPANY,DISTRIBUTOR,COUNTRY_ORIGIN,FILMING_LOCATION,ID_PRODUCTION
0,20th Century Studios,20th Century Studios,United States,"Medfield State Hospital 45 Hospital Road, Med...",1
1,Amen Ra Films,New Line Cinema,United States,"Vancouver, British Columbia, Canada",2
2,Animal Logic,Warner Bros.,United States,"Disney Studios, Moore Park, Sydney, New South ...",3


In [18]:
dim_date = df_silver[["US_RELEASE_DATE"]].dropna().drop_duplicates().sort_values(by="US_RELEASE_DATE").reset_index(drop=True)

dim_date["YEAR"] = dim_date["US_RELEASE_DATE"].dt.year
dim_date["MONTH"] = dim_date["US_RELEASE_DATE"].dt.month
dim_date["QUARTER"] = dim_date["US_RELEASE_DATE"].dt.quarter

dim_date["ID_DATE"] = dim_date["US_RELEASE_DATE"].dt.strftime("%Y%m%d").astype(int)

dim_date.head(2)

,US_RELEASE_DATE,YEAR,MONTH,QUARTER,ID_DATE
0,1978-12-15,1978,12,4,19781215
1,1981-06-19,1981,6,2,19810619


In [19]:
fact_movies = df_silver.copy()

In [20]:
fact_movies = fact_movies.merge(
    dim_film,
    on=["FILM", "GENRE", "RATING_MPA", "LANGUAGE", "BREAK_EVEN", "MCU", "MALE_FEMALE_LED", "PHASE"],
    how="left",
)

In [21]:
fact_movies = fact_movies.merge(
    dim_staff,
    on=["DIRECTOR", "WRITER", "STAR"],
    how="left",
)

In [22]:
fact_movies = fact_movies.merge(
    dim_staff,
    on=["DIRECTOR", "WRITER", "STAR"],
    how="left",
)

In [23]:
fact_movies = fact_movies.merge(
    dim_franchise,
    on=["FRANCHISE", "CHARACTER_FAMILY"],
    how="left",
)

In [24]:
fact_movies = fact_movies.merge(
    dim_production,
    on=["PRODUCTION_COMPANY", "DISTRIBUTOR", "COUNTRY_ORIGIN", "FILMING_LOCATION"],
    how="left",
)

In [25]:
fact_movies = fact_movies.merge(
    dim_date,
    on=["US_RELEASE_DATE"],
    how="left",
)

In [ ]:
fact_movies = fact_movies[
    [
        "ID_FILM", "ID_STAFF", "ID_FRANCHISE", "ID_PRODUCTION",
        "ID_DATE", "BUDGET_x", "INFLATION_ADJUSTED_BUDGET",
        "GROSS_WORLD_WIDE", "INFLATION_ADJUSTED_WORLDWIDE_GROSS",
        "GROSS_US_CANADA", "DOMESTIC", "BOX_OFFICE_GROSS_OTHER_TERRITORIES",
        "GROSS_OPENING_WEEKEND", "GROSS_TO_BUDGET",
        "RATING_IMDB", "ROTTEN_TOMATOES_CRITIC_SCORE",
        "VOTE", "OSCAR", "NOMINATION", "WIN", "MINUTES"
    ]
]

In [26]:
fact_movies.head(3)

,BOX_OFFICE_GROSS_OTHER_TERRITORIES,BREAK_EVEN,BUDGET_x,CHARACTER_FAMILY,COUNTRY_ORIGIN,DIRECTOR,DISTRIBUTOR,DOMESTIC,FILM,FILMING_LOCATION,FRANCHISE,GENRE,GROSS_OPENING_WEEKEND,GROSS_TO_BUDGET,GROSS_US_CANADA,GROSS_WORLD_WIDE,ID,INFLATION_ADJUSTED_BUDGET,INFLATION_ADJUSTED_WORLDWIDE_GROSS,LANGUAGE,LINK,MALE_FEMALE_LED,MCU,MINUTES,NOMINATION,OSCAR,PHASE,PRODUCTION_COMPANY,RATING_IMDB,RATING_MPA,ROTTEN_TOMATOES_CRITIC_SCORE,STAR,US_RELEASE_DATE,VOTE,WIN,WRITER,YEAR_x,int64_field_0,ID_FILM,ID_STAFF_x,ID_STAFF_y,ID_FRANC,ID_PRODUCTION,YEAR_y,MONTH,QUARTER,ID_DATE
0,121200000.0,Flop,84500000.0,Birds of Prey,NaN,NaN,Warner Bros.,0.41,Birds of Prey (and the Fantabulous Emancipatio...,NaN,DC,NaN,NaN,2.43,NaN,NaN,NaN,99482958.0,2.417881e+08,NaN,NaN,Female,False,109.0,0,0,NaN,NaN,NaN,NaN,78,NaN,2020-02-07,0,0,NaN,2020,88,16,104,104,3,107,2020,2,1,20200207
1,186597000.0,Flop,200000000.0,X-Men,NaN,NaN,Twentieth Century Fox,0.26,Dark Phoenix,NaN,Marvel,NaN,NaN,1.26,NaN,NaN,NaN,200000000.0,2.524430e+08,NaN,NaN,Female,False,113.0,0,0,NaN,NaN,NaN,NaN,22,NaN,2019-06-07,0,0,NaN,2019,85,33,104,104,38,106,2019,6,2,20190607
2,741395911.0,Success,160000000.0,Spider-Man,NaN,NaN,Sony Pictures Entertainment,0.35,Spider-Man: Far From Home,NaN,Marvel,NaN,NaN,7.07,NaN,NaN,NaN,160000000.0,1.131928e+09,NaN,NaN,Male,True,129.0,0,0,3.0,NaN,NaN,NaN,90,NaN,2019-07-02,0,0,NaN,2019,86,69,104,104,35,105,2019,7,3,20190702


In [27]:
minhas_tabelas = {
    "dim_date": dim_date,
    "dim_film": dim_film,
    "dim_franchise": dim_franchise,
    "dim_production": dim_production,
    "dim_staff": dim_staff,
    "fact_movies": fact_movies,
}

for key, value in minhas_tabelas.items():
    print(key)

    pandas_gbq.to_gbq(
        value,
        project_id=gcp_id_do_projeto,
        destination_table=f"gold.{key}",
        if_exists="replace",
        credentials=my_gcp_cred,
    )

dim_date
dim_film
dim_franchise
dim_production
dim_staff
fact_movies


In [28]:
minhas_tabelas

{'dim_date':     US_RELEASE_DATE  YEAR  MONTH  QUARTER   ID_DATE
 0        1978-12-15  1978     12        4  19781215
 1        1981-06-19  1981      6        2  19810619
 2        1983-06-17  1983      6        2  19830617
 3        1984-11-21  1984     11        4  19841121
 4        1986-08-01  1986      8        3  19860801
 ..              ...   ...    ...      ...       ...
 108      2023-06-16  2023      6        2  20230616
 109      2023-08-18  2023      8        3  20230818
 110      2023-11-10  2023     11        4  20231110
 111      2023-12-20  2023     12        4  20231220
 112      2024-02-14  2024      2        1  20240214
 
 [113 rows x 5 columns],
 'dim_film':                                   FILM                    GENRE RATING_MPA  \
 0                              Ant-Man                Superhero      PG-13   
 1                 Ant-Man and the Wasp                Superhero      PG-13   
 2    Ant-Man and the Wasp: Quantumania                Superhero      PG-13 